[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [LightGBM and XGBoost, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)

# The scikit-learn Interface


## What you will be able to do

Use LightGBM's and XGBoost's scikit-learn classes as you would any scikit-learn model: fit, predict
and score them, read and change their settings, and run them through cross-validation and a pipeline.

## The idea

### The problem

The **Why Gradient Boosting** notebook trained both libraries in three lines each. A project asks
more of a model than that. You compare settings, cross-validate on folds of stations, put a step in
front of the model that adds a column, and use the same code again for a classifier. The
**scikit-learn, Deep Dive** guide built all of that from one small interface, and every tool in it
expects a model that follows it.

LightGBM and XGBoost each have an interface of their own, which the **Native Interfaces** notebook
covers, written for training a model from a dictionary of settings. Beside it, each library ships
classes that follow scikit-learn's rules, so a boosted model can go wherever a scikit-learn model
goes.

### A formal definition

> A **scikit-learn wrapper** is a class that holds a library's own model behind scikit-learn's
> estimator interface. Its settings are arguments to the constructor, `fit(X, y)` trains, `predict(X)`
> answers, and what was learned is kept in attributes whose names end in an underscore. This guide
> uses four: `LGBMRegressor` and `LGBMClassifier` from LightGBM, and `XGBRegressor` and
> `XGBClassifier` from XGBoost.

### Why it works that way

Because the settings live in the constructor, `get_params`, `set_params` and `clone` work, and
cross-validation uses those three to copy a model, or a pipeline that ends in one, and fit the copy on
each fold. The
model a wrapper trains is kept inside it, as a `Booster`, the object the native interface works
with.

The two libraries agree on what most settings mean and disagree on names and defaults. LightGBM's
`LGBMRegressor()` holds its defaults in the constructor: 100 trees, a learning rate of 0.1, up to 31
leaves a tree. `XGBRegressor()` holds `None` for nearly every setting, which means "let the library
decide", so its `get_params()` cannot tell you what a fit will use. The values come from inside
XGBoost, and the trained booster's configuration records them.

A classifier adds `predict_proba`, which returns a probability for each class, and `classes_`, the
labels in the order of those columns. The two libraries differ there too. XGBoost wants the classes
numbered 0, 1, 2 and so on, and LightGBM accepts any labels.

Both wrappers remember the names of the columns they were trained on. XGBoost compares them with the
columns it is asked to predict from. LightGBM, by default, goes by position, which is where the
second of this notebook's Common errors comes from.

### Where this shows up

The **Tuning** notebook searches over settings with cross-validation, and the **Cross-Validation**
and **Pipelines** notebooks in the **scikit-learn, Deep Dive** guide explain the tools used here.

### What this notebook covers

- fitting, predicting and scoring both regressors
- the settings each constructor holds, and the ones XGBoost actually used
- `set_params`, `clone`, and what a fitted model keeps
- both classifiers, with `classes_` and `predict_proba`
- cross-validation by station, and a model at the end of a pipeline
- three mistakes with columns and classes

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import json
import numpy as np
import xgboost as xgb

model = xgb.XGBRegressor()
print(model.get_params()["learning_rate"])
model.fit(np.arange(10.0).reshape(-1, 1), np.arange(10.0))
config = json.loads(model.get_booster().save_config())
print(config["learner"]["gradient_booster"]["tree_train_param"]["eta"])
```

```
None
0.300000012
```

The constructor kept `None` for the learning rate, and the booster's configuration shows the value
XGBoost used, 0.3, which it calls `eta` and stores as a 32-bit float.

## Setup

Two libraries for the table, the two boosting libraries, and the parts of scikit-learn this notebook
puts them through.

- `json` reads XGBoost's configuration, which a booster returns as text
- `numpy` and `pandas` build the table
- `lightgbm` and `xgboost` are the two libraries, and the cell prints their versions
- `clone` copies a model's settings without what it learned
- `GroupKFold`, `KFold` and `cross_val_score` cross-validate, by station and by row
- `make_pipeline` and `FunctionTransformer` put a step in front of a model
- `mean_squared_error` scores the regressors on the test and validation stations

`station_days` makes the same 20,000 made-up station-days as in the **Why Gradient Boosting**
notebook. Each row is one day at one of 400 weather stations in Norway, with the station's latitude,
elevation, distance from the coast and terrain, the day's cloud, wind and snow, and `tmin`, the day's
lowest temperature. A generator started from a fixed seed draws them, so they are the same on every
run. The cell splits the stations into 300 for training, 50 for validation and 50 for testing,
keeping each station's days together. `FEATURES` names the eight columns a model is given.

In [1]:
import json

import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from sklearn.base import clone
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GroupKFold, KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer

TERRAINS = ["coastal", "fjord", "inland", "valley", "mountain"]


def station_days(n_stations=400, days=50, seed=2026):
    """Made-up daily minimum temperatures at weather stations in Norway, one row per station and day."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n_stations)
    coast_km = np.minimum(rng.exponential(30, n_stations), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n_stations) + 2.5 * coast_km, 0, 1600)
    terrain = rng.choice(TERRAINS, n_stations, p=[0.2, 0.2, 0.25, 0.2, 0.15])
    has_snow_sensor = rng.uniform(size=n_stations) < 0.7
    offset = rng.normal(0, 0.8, n_stations)
    station = np.repeat(np.arange(n_stations), days)
    n = n_stations * days
    day = rng.integers(1, 366, n)
    cloud = np.clip(rng.beta(1.2, 1.0, n), 0, 1)
    wind = rng.gamma(2.0, 2.0, n)
    season = np.cos(2 * np.pi * (day - 20) / 365)
    lat, elev, coast = latitude[station], elevation[station], coast_km[station]
    snow = np.where(season > 0.2, np.clip(rng.normal(30 * season + 0.03 * elev + 2 * (lat - 58), 15), 0, None), 0.0)
    shelter = pd.Series(terrain[station]).map({"coastal": 0.4, "fjord": 1.0, "inland": 1.2, "valley": 1.7, "mountain": 0.8}).to_numpy()
    mean = 8 - 0.55 * (lat - 58) - 0.006 * elev - 9 * season + 3.5 * np.exp(-coast / 40) * season
    cooling = 7 * (1 - cloud) * np.exp(-wind / 3) * (0.4 + (season + 1) / 2) * shelter
    tmin = mean - 2.5 - cooling - 2.0 * (snow > 10) * (1 - cloud) + offset[station] + rng.normal(0, 1.2, n)
    snow_cm = np.where(has_snow_sensor[station], snow.round(0), np.nan)
    return pd.DataFrame({
        "station": [f"ST{s + 1:04d}" for s in station],
        "latitude": lat.round(2), "elevation_m": elev.round(0).astype(int), "coast_km": coast.round(1),
        "terrain": pd.Categorical(terrain[station], categories=TERRAINS),
        "day_of_year": day, "cloud_cover": cloud.round(2), "wind_ms": wind.round(1), "snow_cm": snow_cm,
        "tmin": tmin.round(1),
    })


FEATURES = ["latitude", "elevation_m", "coast_km", "terrain", "day_of_year", "cloud_cover", "wind_ms", "snow_cm"]
days = station_days()
stations = days["station"].unique()
order = np.random.default_rng(0).permutation(len(stations))
train = days[days["station"].isin(stations[order[:300]])]
valid = days[days["station"].isin(stations[order[300:350]])]
test = days[days["station"].isin(stations[order[350:]])]

print("lightgbm", lgb.__version__, "| xgboost", xgb.__version__, "|", len(train), "training rows,",
      len(valid), "validation rows,", len(test), "test rows")

lightgbm 4.6.0 | xgboost 3.4.1 | 15000 training rows, 2500 validation rows, 2500 test rows


## Worked examples

### Fit, predict and score

The cell trains both regressors with their defaults on the training stations and scores them on the
test stations. `score` returns R², the share of the variation in the answers that the model explains.

In [2]:
y_train, y_test = train["tmin"], test["tmin"]

lightgbm_model = lgb.LGBMRegressor(random_state=0, verbose=-1).fit(train[FEATURES], y_train)
xgboost_model = xgb.XGBRegressor(random_state=0).fit(train[FEATURES], y_train)

for name, model in [("LightGBM", lightgbm_model), ("XGBoost", xgboost_model)]:
    print(name, "| first three predictions", model.predict(test[FEATURES])[:3].round(1),
          "| R²", round(model.score(test[FEATURES], y_test), 3),
          "| mean squared error", round(mean_squared_error(y_test, model.predict(test[FEATURES])), 3))

LightGBM | first three predictions [ 0.2 -2.4  8.9] | R² 0.943 | mean squared error 2.555
XGBoost | first three predictions [ 0.2 -2.   9. ] | R² 0.936 | mean squared error 2.883


Both models use their libraries' defaults. In this run LightGBM's test loss is 2.555 and XGBoost's
2.883, and their R² is 0.943 and 0.936, so each explains more than nine tenths of the variation in
the day's lowest temperature.

### The settings each constructor holds

`get_params` returns every setting of the constructor by name. The two classes share some names and
not others, and `-` in the table below marks a setting the class does not have.

In [3]:
lightgbm_params = lgb.LGBMRegressor().get_params()
xgboost_params = xgb.XGBRegressor().get_params()
for setting in ["n_estimators", "learning_rate", "num_leaves", "max_leaves", "max_depth", "subsample", "reg_lambda"]:
    print(f"{setting:14} {str(lightgbm_params.get(setting, '-')):6} {xgboost_params.get(setting, '-')}")

n_estimators   100    None
learning_rate  0.1    None
num_leaves     31     -
max_leaves     -      None
max_depth      -1     None
subsample      1.0    None
reg_lambda     0.0    None


LightGBM's constructor holds its defaults: 100 trees, a learning rate of 0.1, 31 leaves, no limit on
depth (`-1`), every row for every tree (`subsample` 1.0) and no penalty on leaf values. XGBoost's
holds `None` for each of them, and has no `num_leaves`. By default its trees are limited by depth,
and its own limit on leaves is called `max_leaves`.

### The settings XGBoost used

A trained XGBoost booster returns its whole configuration as JSON text. The tree settings sit under
`learner`, `gradient_booster` and `tree_train_param`, and `num_boosted_rounds` counts the trees.

In [4]:
config = json.loads(xgboost_model.get_booster().save_config())
used = config["learner"]["gradient_booster"]["tree_train_param"]
print({setting: used[setting] for setting in ["eta", "max_depth", "lambda", "subsample"]})
print("trees:", xgboost_model.get_booster().num_boosted_rounds())
print("LightGBM trees:", lightgbm_model.booster_.num_trees())

{'eta': '0.300000012', 'max_depth': '6', 'lambda': '1', 'subsample': '1'}
trees: 100
LightGBM trees: 100


XGBoost used a learning rate of 0.3, stored as the 32-bit float `0.300000012`, a depth of 6, a
penalty `lambda` of 1 and every row, for 100 trees, the number LightGBM's default gives too. The two
defaults differ in the learning rate, the size of a tree and the penalty, and the solutions try
XGBoost at LightGBM's learning rate.

### Changing a setting and copying a model

`set_params` changes settings in place and returns the model, and `clone` makes a new, unfitted model
with the same settings, which is how cross-validation gets a fresh model for every fold.

In [5]:
copy = clone(lightgbm_model).set_params(n_estimators=50)
print("copy:", copy.get_params()["n_estimators"], "trees, fitted:", hasattr(copy, "n_features_in_"))
print("original:", lightgbm_model.get_params()["n_estimators"], "trees, fitted:",
      hasattr(lightgbm_model, "n_features_in_"))

copy: 50 trees, fitted: False
original: 100 trees, fitted: True


The copy has 50 trees in its settings and has learned nothing, and the original keeps its 100.
`n_features_in_` exists only after `fit`, which makes it a quick test of whether a model has been
fitted.

### What a fitted model keeps

Both wrappers record the columns they were trained on under `feature_names_in_`, scikit-learn's own
name, and LightGBM keeps them under `feature_name_` too.

In [6]:
print("LightGBM:", lightgbm_model.n_features_in_, [str(name) for name in lightgbm_model.feature_names_in_])
print("XGBoost: ", xgboost_model.n_features_in_, [str(name) for name in xgboost_model.feature_names_in_])
print("LightGBM's feature_name_ holds the same names:",
      lightgbm_model.feature_name_ == list(lightgbm_model.feature_names_in_))

LightGBM: 8 ['latitude', 'elevation_m', 'coast_km', 'terrain', 'day_of_year', 'cloud_cover', 'wind_ms', 'snow_cm']
XGBoost:  8 ['latitude', 'elevation_m', 'coast_km', 'terrain', 'day_of_year', 'cloud_cover', 'wind_ms', 'snow_cm']
LightGBM's feature_name_ holds the same names: True


Both models kept the eight column names in the order of `FEATURES`, and LightGBM's `feature_name_`
holds the same list. `feature_names_in_` holds them as NumPy strings, which the cell turns into plain
ones to print.

### Classifiers

`frost` is 1 for a day whose lowest temperature is below 0 and 0 otherwise. The classifiers take the
same table and the same settings as the regressors.

In [7]:
frost_train = (train["tmin"] < 0).astype(int)
frost_test = (test["tmin"] < 0).astype(int)

lightgbm_classifier = lgb.LGBMClassifier(random_state=0, verbose=-1).fit(train[FEATURES], frost_train)
xgboost_classifier = xgb.XGBClassifier(random_state=0).fit(train[FEATURES], frost_train)

for name, model in [("LightGBM", lightgbm_classifier), ("XGBoost", xgboost_classifier)]:
    print(name, "| classes", model.classes_, "| first five", model.predict(test[FEATURES])[:5],
          "| accuracy", round(model.score(test[FEATURES], frost_test), 3))
    print(model.predict_proba(test[FEATURES])[:3].round(3))

LightGBM | classes [0 1] | first five [0 1 0 0 0] | accuracy 0.936
[[0.869 0.131]
 [0.041 0.959]
 [0.999 0.001]]
XGBoost | classes [0 1] | first five [0 1 0 0 0] | accuracy 0.936
[[0.875 0.125]
 [0.069 0.931]
 [1.    0.   ]]


Both classify 93.6% of the test days correctly and agree on the first five. Each row of
`predict_proba` holds the probability of no frost and of frost, in the order of `classes_`, so
`[:, 1]` is the probability of frost. On these three days the two libraries' probabilities differ by
about 0.03 at most.

### Cross-validation by station

`cross_val_score` clones the model for each fold, fits it on four fifths of the training rows and
scores it on the rest. `GroupKFold` keeps every station's days in one fold, as the split in Setup
does, and `shuffle=True` with a `random_state` makes it split a seeded permutation of the stations.
Without `shuffle`, it gives the stations to the folds largest first. Every training station has 50
days, so the order they go in comes from how NumPy's `argsort` orders equal values, which NumPy leaves
unspecified. NumPy 2.1.3 and 2.4.6 ordered them differently, and so made different folds. `KFold` with `shuffle=True` splits the rows
at random, so most of a station's days train the model that is scored on its other days.

In [8]:
station_folds = GroupKFold(n_splits=5, shuffle=True, random_state=0)
by_station = cross_val_score(lgb.LGBMRegressor(random_state=0, verbose=-1), train[FEATURES], y_train,
                             groups=train["station"], cv=station_folds, scoring="neg_mean_squared_error")
by_row = cross_val_score(lgb.LGBMRegressor(random_state=0, verbose=-1), train[FEATURES], y_train,
                         cv=KFold(n_splits=5, shuffle=True, random_state=0),
                         scoring="neg_mean_squared_error")
print("folds of stations:", (-by_station).round(3), "| mean", round(-by_station.mean(), 3))
print("folds of rows:    ", (-by_row).round(3), "| mean", round(-by_row.mean(), 3))

folds of stations: [2.714 2.462 2.752 2.654 3.357] | mean 2.788
folds of rows:     [1.917 1.86  1.805 1.868 1.928] | mean 1.876


Folds of stations give losses from 2.462 to 3.357, with a mean of 2.788. Folds of rows give a mean of
1.876, much lower, because a station's own warmth or coldness is learned from its other days. Folds of
stations measure what the test stations measure, stations the model has never seen, and folds of rows
promise more than a new station will get.

### A model at the end of a pipeline

`FunctionTransformer` turns a function into a pipeline step. `add_season` adds the cosine of the day
of the year, cold at both ends of the year, as the linear model in the **Why Gradient Boosting**
notebook had it. The comparison with and without the column is a choice between models, so the cell
makes it on the validation stations and leaves the test stations alone.

In [9]:
def add_season(rows):
    return rows.assign(season=np.cos(2 * np.pi * (rows["day_of_year"] - 20) / 365))

pipeline = make_pipeline(FunctionTransformer(add_season), lgb.LGBMRegressor(random_state=0, verbose=-1))
pipeline.fit(train[FEATURES], y_train)
y_valid = valid["tmin"]
print("with season:", round(mean_squared_error(y_valid, pipeline.predict(valid[FEATURES])), 3))
print("without:", round(mean_squared_error(y_valid, lightgbm_model.predict(valid[FEATURES])), 3))
print(pipeline[-1].feature_name_)

with season: 2.489
without: 2.553
['latitude', 'elevation_m', 'coast_km', 'terrain', 'day_of_year', 'cloud_cover', 'wind_ms', 'snow_cm', 'season']


With the season column the validation loss is 2.489, against 2.553 without it, from one split and
one seed, which is too little to say the column helps. The model at the end of the pipeline lists
`season` among its nine features, so the step ran before it, at fitting and at predicting.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/02-the-scikit-learn-interface-solutions.ipynb).

**1.** Print the `n_estimators` and `learning_rate` that `get_params()` holds for `LGBMClassifier()` and for `XGBClassifier()`.

In [10]:
# your code here

**2.** Fit an `XGBRegressor` with `learning_rate=0.1`, LightGBM's default, keeping its other defaults, and print its validation loss beside the default `xgboost_model`'s.

In [11]:
# your code here

**3.** Cross-validate `XGBRegressor(random_state=0)` by station with `GroupKFold(n_splits=5, shuffle=True, random_state=0)`, and print the five losses and their mean.

In [12]:
# your code here

**4.** Put `FunctionTransformer(add_season)` in front of an `XGBRegressor(random_state=0)` and cross-validate the pipeline by station, then compare its mean loss with the previous task's.

In [13]:
# your code here

**5.** Count how many test days each classifier gives a frost probability above 0.9, and what share of those days really had frost.

In [14]:
# your code here

## Common errors

### `ValueError: feature_names mismatch: ['cloud_cover', 'wind_ms'] ['wind_ms', 'cloud_cover']`

An XGBoost model trained on two columns is asked to predict from a table that holds the same two
columns in the other order.

In [15]:
two_columns = ["cloud_cover", "wind_ms"]
small_xgboost = xgb.XGBRegressor(random_state=0).fit(train[two_columns], y_train)
small_xgboost.predict(test[["wind_ms", "cloud_cover"]])

ValueError: feature_names mismatch: ['cloud_cover', 'wind_ms'] ['wind_ms', 'cloud_cover']

XGBoost compares the names it was trained on, the first list, with the names it is given, the second,
and raises for a difference in order. Select the columns in the order `feature_names_in_` holds them.

In [16]:
print(round(mean_squared_error(y_test, small_xgboost.predict(test[list(small_xgboost.feature_names_in_)])), 3))

47.484


In the order it knows, the model predicts with a test loss of 47.484, higher than the 44.918 of
predicting the mean in the **Why Gradient Boosting** notebook.

### No error, and wrong predictions from columns in another order

A LightGBM model of its own, trained on the eight columns in the order of `FEATURES`, is asked to
predict from the same columns sorted by name.

In [17]:
positional_model = lgb.LGBMRegressor(random_state=0, verbose=-1).fit(train[FEATURES], y_train)
sorted_columns = test[sorted(FEATURES)]
print(list(sorted_columns.columns))
print(round(mean_squared_error(y_test, positional_model.predict(sorted_columns)), 3))

['cloud_cover', 'coast_km', 'day_of_year', 'elevation_m', 'latitude', 'snow_cm', 'terrain', 'wind_ms']
86.405


The table holds the same eight columns, sorted by name, and LightGBM predicted from them by position,
so `cloud_cover` went where the model expected `latitude`, and so on. The test loss is 86.405, nearly
twice the 44.918 of predicting the mean, and nothing warned.

In [18]:
try:
    positional_model.predict(sorted_columns, validate_features=True)
except lgb.basic.LightGBMError as error:
    print("LightGBMError:", error)

LightGBMError: Expected 'latitude' at position 0 but found 'cloud_cover'


[LightGBM] [Fatal] Expected 'latitude' at position 0 but found 'cloud_cover'


`validate_features=True` makes LightGBM compare the names before it predicts, and it raises a
`LightGBMError` that names the first column out of place, after printing the same message as a
`[Fatal]` line. Select `test[FEATURES]` before predicting, or pass `validate_features=True` where a
table could arrive with its columns in another order.

### ``ValueError: Invalid classes inferred from unique values of `y`.  Expected: [0 1 2], got [1 2 3]``

The cell numbers bands of cold 1, 2 and 3, for days at −10 or below, above −10 and up to 0, and above
0, and fits `XGBClassifier` to them.

In [19]:
bands = pd.cut(train["tmin"], [-100, -10, 0, 100], labels=False) + 1
print(bands.value_counts().sort_index().to_dict())
xgb.XGBClassifier(random_state=0).fit(train[FEATURES], bands)

{1: 1654, 2: 6627, 3: 6719}


ValueError: Invalid classes inferred from unique values of `y`.  Expected: [0 1 2], got [1 2 3]

The bands hold 1,654 training days at −10 or below, 6,627 above −10 and up to 0, and 6,719 above 0.
XGBoost numbers a classifier's classes from 0, so it expects the labels 0, 1 and 2 and names the ones it found.

In [20]:
banded = xgb.XGBClassifier(random_state=0).fit(train[FEATURES], bands - 1)
print("XGBoost:", banded.classes_, banded.predict(test[FEATURES])[:5] + 1)
lightgbm_banded = lgb.LGBMClassifier(random_state=0, verbose=-1).fit(train[FEATURES], bands)
print("LightGBM:", lightgbm_banded.classes_, lightgbm_banded.predict(test[FEATURES])[:5])

XGBoost: [0 1 2] [3 2 3 3 3]
LightGBM: [1 2 3] [3 2 3 3 3]


Subtracting 1 gives XGBoost the labels it expects, and adding 1 to its predictions turns them back
into bands. LightGBM takes the bands as they are and keeps them in `classes_`. Both predict the same
bands for the first five test days.

## Recap

- You can fit, predict and score `LGBMRegressor`, `XGBRegressor` and their classifiers as scikit-learn
  models.
- You can read the settings a constructor holds, and read from XGBoost's booster the ones it used.
- You can cross-validate a boosted model by station and put it at the end of a pipeline.
- You can give a model its columns in the order it was trained on, and give XGBoost classes numbered
  from 0.

## What is next

The **Native Interfaces** notebook trains the same models from LightGBM's `Dataset` and XGBoost's
`DMatrix`, with the settings in a dictionary.

---

&#8592; **Previous:** [Why Gradient Boosting](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/01-why-gradient-boosting.ipynb)  &nbsp;·&nbsp;  [LightGBM and XGBoost, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [The Native Interfaces](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/03-the-native-interfaces.ipynb) &#8594;
